# 08. Bitmask DP, Digit DP and Space Optimization

Harder DP shapes. Bitmask DP works when `n ≤ 20`. Digit DP counts numbers with a property. Space optimization keeps only the rows you need.

**Learning order:** BITMASK STATE → TSP → BFS WITH MASK → PARTITION K SUBSETS → DICTIONARY STATE → DIGIT DP → SPACE OPTIMIZATION → INTERVIEW THINKING

## 1. Bitmask DP: the mask is the state

When you must remember **which items are already used** (up to ~20), store them as bits of an integer. `dp[mask]` or `dp[mask][last]`. There are `2ⁿ` masks. Real life: a delivery person visiting every city once (**Traveling Salesman Problem**).

In [1]:
def tsp(dist):
    n = len(dist)
    INF = float("inf")
    # dp[mask][i] = min cost to visit the set `mask`, ending at city i, starting at city 0
    dp = [[INF] * n for _ in range(1 << n)]
    dp[1][0] = 0
    for mask in range(1 << n):
        for last in range(n):
            if dp[mask][last] == INF or not (mask >> last) & 1:
                continue
            for nxt in range(n):
                if (mask >> nxt) & 1:
                    continue                                   # already visited
                new_mask = mask | (1 << nxt)
                dp[new_mask][nxt] = min(dp[new_mask][nxt], dp[mask][last] + dist[last][nxt])
    full = (1 << n) - 1
    return min(dp[full][i] + dist[i][0] for i in range(1, n))     # return to the start

d = [[0, 10, 15, 20], [10, 0, 35, 25], [15, 35, 0, 30], [20, 25, 30, 0]]
print(tsp(d))   # 80

80


Complexity: `2ⁿ × n` states, `n` transitions each → **O(2ⁿ · n²)**, far better than `n!` permutations.

## 2. BFS with a mask: Shortest Path Visiting All Nodes

State = `(current node, set of visited nodes)`. BFS from **every node at once**. The first time the mask is "all visited", the number of steps is the answer. Mark `(node, mask)` as seen.

In [2]:
from collections import deque

def shortest_path_length(graph):
    n = len(graph)
    full = (1 << n) - 1
    q = deque((i, 1 << i, 0) for i in range(n))
    seen = {(i, 1 << i) for i in range(n)}
    while q:
        node, mask, steps = q.popleft()
        if mask == full:
            return steps
        for nb in graph[node]:
            nm = mask | (1 << nb)
            if (nb, nm) not in seen:
                seen.add((nb, nm))
                q.append((nb, nm, steps + 1))
    return 0

print(shortest_path_length([[1, 2, 3], [0], [0], [0]]), shortest_path_length([[1], [0, 2, 4], [1, 3, 4], [2], [1, 2]]))   # 4 4

4 4


## 3. Partition to K Equal Sum Subsets (mask DP)

Better than backtracking when `n ≤ 16`. `total[mask]` = sum of chosen numbers. State: `dp[mask]` = the **current bucket fill** (sum modulo target) after using the numbers in `mask`, or `-1` if impossible.

In [3]:
def can_partition_k_subsets(nums, k):
    total = sum(nums)
    if total % k:
        return False
    target = total // k
    n = len(nums)
    dp = [-1] * (1 << n)                  # dp[mask] = sum of current bucket, -1 = unreachable
    dp[0] = 0
    for mask in range(1 << n):
        if dp[mask] == -1:
            continue
        for i in range(n):
            if mask >> i & 1:
                continue
            if dp[mask] + nums[i] <= target:
                new = mask | (1 << i)
                dp[new] = (dp[mask] + nums[i]) % target
    return dp[(1 << n) - 1] == 0

print(can_partition_k_subsets([4, 3, 2, 3, 5, 2, 1], 4), can_partition_k_subsets([1, 2, 3, 4], 3))   # True False

True False


## 4. A dictionary as the state: Longest Arithmetic Subsequence

State: `(index i, common difference d)`. `dp[i][d]` = length of the longest arithmetic subsequence ending at `i` with difference `d`. Differences can be large, so use a **dict per index** instead of a table.

In [4]:
def longest_arith_seq_length(nums):
    dp = [dict() for _ in nums]
    best = 2
    for i in range(len(nums)):
        for j in range(i):
            d = nums[i] - nums[j]
            dp[i][d] = dp[j].get(d, 1) + 1
            best = max(best, dp[i][d])
    return best

print(longest_arith_seq_length([3, 6, 9, 12]), longest_arith_seq_length([9, 4, 7, 2, 10]))   # 4 3

4 3


## 5. Digit DP: count numbers ≤ N with a property

Build the number **digit by digit** from the left. State: `(position, mask of digits used, tight, started)`.
- `tight`: so far equal to the prefix of `N` (the next digit is limited by `N`'s digit).
- `started`: leading zeros are not counted as digits.

Example: **count integers in `[1, n]` whose digits are all distinct**.

In [5]:
from functools import lru_cache

def count_special_integers(n):
    digits = list(map(int, str(n)))
    @lru_cache(maxsize=None)
    def dp(i, mask, tight, started):
        if i == len(digits):
            return 1 if started else 0
        limit = digits[i] if tight else 9
        total = 0
        for d in range(limit + 1):
            if started and (mask >> d) & 1:
                continue                                     # digit already used
            new_started = started or d != 0
            new_mask = mask | (1 << d) if new_started else mask
            total += dp(i + 1, new_mask, tight and d == limit, new_started)
        return total
    return dp(0, 0, True, False)

print(count_special_integers(20), count_special_integers(5), count_special_integers(135))   # 19 5 110

19 5 110


## 6. Space optimization

If `dp[i]` depends only on the previous row/few values, keep just those.

| Pattern | Keep |
|---|---|
| `dp[i] = f(dp[i-1], dp[i-2])` | two variables |
| `dp[i][j]` from the row above | one row (or two) |
| 0/1 knapsack | one array, loop downwards |
| LCS/edit distance | two rows (or one row + a saved diagonal) |

**Trade-off:** you can no longer **reconstruct the answer** (the actual subsequence/path) from a compressed table.

In [6]:
def lcs_two_rows(a, b):
    if len(a) < len(b):
        a, b = b, a                                           # make b the shorter one
    prev = [0] * (len(b) + 1)
    for ch in a:
        cur = [0]
        for j in range(1, len(b) + 1):
            cur.append(prev[j - 1] + 1 if ch == b[j - 1] else max(prev[j], cur[j - 1]))
        prev = cur
    return prev[-1]

print(lcs_two_rows("abcde", "ace"), lcs_two_rows("abc", "def"))   # 3 0

3 0


## 7. Interview thinking

**Signals:** "visit every node/city once", "assign n ≤ 16 items", "partition into k groups, n ≤ 16", "count numbers up to N with digit restrictions", "subsequence where the difference matters".

**Say:** "The state needs the set of used items, so I use a bitmask. `2ⁿ` states × `n` transitions. n ≤ 20 makes this feasible."

**Common mistakes:** mask operations without parentheses, forgetting to return to the start (TSP), `2ⁿ` memory when `n` is too large, not handling leading zeros in digit DP, forgetting the `tight` flag.

## Quick Revision

- Bitmask DP: `dp[mask]` or `dp[mask][last]`, `n ≤ 20`, O(2ⁿ·n²) for TSP.
- BFS over `(node, mask)` for "visit all nodes".
- Dict state when the second parameter is large (arithmetic difference).
- Digit DP: `(pos, mask, tight, started)`.
- Space optimization: rolling rows, but you lose reconstruction.